# Titanic Survival Prediction
Albert Shan

A curated version of the original Kaggle study. Feature formulas, model settings, and validation split are retained. See `original_kaggle.ipynb` for the source history and `../reports/evaluation.md` for historical results.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.titanic import FEATURES, add_features, build_model
DATA_DIR = ROOT / "data"
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
print("Training data shape:", train.shape)
print("Test data shape:", test.shape)

## 1. Explore the data
Inspect missing values, distributions, and survival rates before fitting models.

In [ ]:
train.head()
train.shape
train.columns
train.info()
train.isnull().sum()

In [ ]:
train.isnull().mean() * 100

In [ ]:
train.describe()

In [ ]:
train["Age"].hist()

In [ ]:
train["Fare"].hist()

In [ ]:
train["Survived"].value_counts(normalize=True)

In [ ]:
train.groupby("Sex")["Survived"].mean()
train.groupby("Pclass")["Survived"].mean()
train.groupby("Survived")["Age"].mean()

In [ ]:
train["AgeGroup"] = pd.cut(
    train["Age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["Child", "Teen", "Young Adult", "Adult", "Senior"]
)

train.groupby("AgeGroup", observed=True)["Survived"].mean()

In [ ]:
train.groupby("SibSp")["Survived"].mean()

In [ ]:

train.groupby("Parch")["Survived"].mean()

In [ ]:
train["Parch"].value_counts().sort_index()

## 2. Engineer family and title features
FamilySize = SibSp + Parch + 1. IsAlone flags a family size of one. Titles are extracted from Name, normalized, and grouped into four common titles and Rare.

In [ ]:
train = add_features(train)
test = add_features(test)
train["Title"].value_counts()

In [ ]:
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
train.groupby("FamilySize")["Survived"].mean()

In [ ]:
train.groupby("IsAlone")["Survived"].mean()

In [ ]:
train.groupby("Embarked")["Survived"].mean()

In [ ]:
train.groupby("Survived")["Fare"].mean()

## 3. Compare models
Use the original stratified 80/20 split (random_state=42). Every model receives a fresh preprocessing pipeline fitted on the training subset.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score

X, y = train[FEATURES], train["Survived"]
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
results = {}
for name in ["baseline_logistic", "logistic", "random_forest", "tuned_random_forest", "xgboost"]:
    model = build_model("logistic" if name == "baseline_logistic" else name,
                        include_title=name != "baseline_logistic")
    model.fit(X_train, y_train)
    results[name] = accuracy_score(y_valid, model.predict(X_valid))
pd.Series(results, name="validation_accuracy")

## 4. Five-fold evaluation
The original `cv=5` classifier evaluation uses stratified folds without shuffling. These folds include rows examined during exploration and tuning, so the result is exploratory rather than an untouched final test estimate.

In [ ]:
model = build_model()
scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
print(scores)
print("Mean:", scores.mean())
print("Std:", scores.std())

## 5. Train on all labeled data and export predictions
Kaggle test labels are unavailable locally. No leaderboard score is claimed.

In [ ]:
model.fit(X, y)
submission = pd.DataFrame({"PassengerId": test["PassengerId"],
                           "Survived": model.predict(test[FEATURES])})
output_dir = ROOT / "artifacts"
output_dir.mkdir(exist_ok=True)
submission.to_csv(output_dir / "submission.csv", index=False)
print(submission.shape)